# Client Satisfaction Regression
This notebook predicts `client_satisfaction_score` and compares multiple regression algorithms.

Le **nettoyage** (doublons, cible manquante, traitement des extrêmes) est réalisé **dans ce notebook** avant le `train_test_split` ; le pipeline sklearn applique ensuite imputation / scaling sur le jeu d'entraînement uniquement.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if (ROOT / 'ml-workspace').exists():
    ROOT = ROOT / 'ml-workspace'
DATA_PATH = ROOT / 'data/raw/modeling_dataset.csv'
RESULTS_PLOTS = ROOT / 'results/plots'
RESULTS_METRICS = ROOT / 'results/metrics'
RESULTS_PLOTS.mkdir(parents=True, exist_ok=True)
RESULTS_METRICS.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
df = pd.read_csv(DATA_PATH)
display(df.head())
print(df['client_satisfaction_score'].describe())
sns.histplot(df['client_satisfaction_score'], bins=30, kde=True)
plt.title('Target Distribution - client_satisfaction_score')
plt.show()

## Nettoyage des données (étapes explicites)
1. Suppression des doublons et des lignes sans score de satisfaction.
2. Imputation simple des manquants sur les colonnes numériques (médiane) et catégorielles (mode) pour traçabilité.
3. **Réduction des extrêmes** sur `deadline_overrun_days`, `avg_task_delay_days`, `budget_usd` par plage IQR (winsorisation / clipping) afin de limiter l'influence des queues de distribution, comme dans le guide de pré-traitement.

In [ ]:
target = 'client_satisfaction_score'
clean_df = df.copy()
n0 = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print(f'Doublons supprimés: {n0 - len(clean_df)} ligne(s)')
clean_df = clean_df.dropna(subset=[target]).copy()
print('Lignes après suppression des NaN sur la cible:', len(clean_df))

null_before = clean_df.isna().sum()
num_cols_all = clean_df.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols_all:
    if clean_df[c].isna().any():
        med = clean_df[c].median()
        nmiss = clean_df[c].isna().sum()
        clean_df[c] = clean_df[c].fillna(med)
        print(f'Imputation médiane {c}: {nmiss} valeur(s)')
cat_cols_all = clean_df.select_dtypes(exclude=[np.number]).columns.tolist()
if target in cat_cols_all:
    cat_cols_all.remove(target)
for c in cat_cols_all:
    if clean_df[c].isna().any():
        mode = clean_df[c].mode(dropna=True)
        fill = mode.iloc[0] if len(mode) else 'unknown'
        clean_df[c] = clean_df[c].fillna(fill)
        print(f'Imputation mode {c}')

clip_cols = ['deadline_overrun_days', 'avg_task_delay_days', 'budget_usd']
bounds = {}
for col in clip_cols:
    q1, q3 = clean_df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    bounds[col] = (lower, upper)
    before = clean_df[col].describe()[['min', 'max']]
    clean_df[col] = clean_df[col].clip(lower, upper)
    after = clean_df[col].describe()[['min', 'max']]
    print(f'{col}: bornes IQR [{lower:.3g}, {upper:.3g}] — min/max avant {before.to_dict()} apres {after.to_dict()}')

fig, axes = plt.subplots(1, len(clip_cols), figsize=(4 * len(clip_cols), 4))
if len(clip_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, clip_cols):
    sns.boxplot(y=clean_df[col], ax=ax, color='#93c5fd')
    ax.set_title(f'Après clipping — {col}')
plt.tight_layout()
plt.show()

null_after = clean_df.isna().sum()
display(pd.DataFrame({'avant': null_before, 'apres': null_after}).fillna(0).astype(int).T)

## Pré-traitement (pipeline + découpage train / test)
Normalisation `MinMaxScaler` + `StandardScaler` sur le bloc numérique et encodage one-hot sur le bloc catégoriel, comme dans le script `run_all_models.py`.

In [ ]:
drop_cols = ['project_id','title','status','success_flag', target]
features = [c for c in clean_df.columns if c not in drop_cols]
X, y = clean_df[features], clean_df[target]

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('normalize', MinMaxScaler()),
        ('standardize', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=SEED)

In [ ]:
models = {
    'linear_regression': LinearRegression(),
    'random_forest_regressor': RandomForestRegressor(n_estimators=260, random_state=SEED),
    'gradient_boosting_regressor': GradientBoostingRegressor(random_state=SEED)
}
rows = []
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    residuals = y_test - pred
    rows.append({
        'algorithm': name,
        'mae': mean_absolute_error(y_test, pred),
        'rmse': np.sqrt(mean_squared_error(y_test, pred)),
        'r2': r2_score(y_test, pred)
    })

    plt.figure(figsize=(6,5))
    plt.scatter(y_test, pred, alpha=0.55)
    lo, hi = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
    plt.plot([lo,hi],[lo,hi],'r--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title(f'Predicted vs Actual - {name}')
    plt.tight_layout()
    plt.savefig(RESULTS_PLOTS / f'regression_pred_vs_actual_{name}.png', dpi=160)
    plt.show()

    fig, axes = plt.subplots(1,2,figsize=(10,4))
    sns.histplot(residuals, kde=True, ax=axes[0])
    axes[0].set_title(f'Residual Distribution - {name}')
    stats.probplot(residuals, dist='norm', plot=axes[1])
    axes[1].set_title(f'QQ Plot - {name}')
    plt.tight_layout()
    plt.savefig(RESULTS_PLOTS / f'regression_residuals_{name}.png', dpi=160)
    plt.show()

metrics_df = pd.DataFrame(rows).sort_values('r2', ascending=False)
display(metrics_df)
metrics_df.to_csv(RESULTS_METRICS / 'regression_metrics.csv', index=False)
metrics_df.to_json(RESULTS_METRICS / 'regression_metrics.json', orient='records', indent=2)
print('Best model based on R2:', metrics_df.iloc[0]['algorithm'])